# ResQue: Warm-Start QRC for Multi-Output Weather Forecasting over East Africa
## ResQue | GIC 2026 | Track B: Weather Time-Series Forecasting

[![Launch on qBraid](https://qbraid-static.s3.amazonaws.com/logos/Launch_on_qBraid_white.png)](https://account.qbraid.com?gitHubUrl=https://github.com/Armstrong66/resque-qrc)

**This notebook is the Phase 3 entry point. Run cells sequentially.**  
All results written to `outputs/results/`. Full benchmark tables match the write-up.

**Cells 1-12 are 100% simulator** (free, no QPU credits used) - that's where
the Hamiltonian/noise/qubit/shot sweeps and baseline training happen. Real
QPU credits are spent ONLY in **Cell 13**, which validates the final
selected config over a small subsampled window - see that cell before
running it.

| Setting | Value |
|---|---|
| Station | NOAA ISD 63450099999 (Addis Ababa Bole, Ethiopia) |
| Horizons | 6h and 24h |
| Primary reservoir | 9-qubit transverse-field Ising chain |
| Encoding ablation | Standard vs. data reuploading (Perez-Salinas et al. 2020) |
| Warm-start | ESN / LSTM / GRU -> QRC via truncated SVD |
| Hardware (Cell 13 only) | QuEra Aquila (PRIMARY) / IBM Eagle (FALLBACK) |

**AI disclosure**: Claude (Anthropic) used for code scaffolding - disclosed per GIC rules.

---
## Cell 1 - Environment setup

In [ ]:
# enforce torch if missing
%pip install torch

In [ ]:
# Install dependencies
import subprocess, sys

def _install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

# Main dependencies
for pkg in ['statsmodels', 'pmdarima', 'pyarrow', 'pennylane-lightning']:
    try:
        __import__(pkg.replace('-', '_').split('==')[0])
    except ImportError:
        print(f'Installing {pkg}...')
        try:
            _install(pkg)
        except Exception as e:
            print(f'  {pkg} install failed ({e})')

# Bloqade-analog for Aquila
try:
    import bloqade.analog
except ImportError:
    print('Installing bloqade-analog...')
    try:
        _install('bloqade-analog')
    except Exception as e:
        print(f'  bloqade-analog install failed ({e})')

# IBM backend for Eagle
try:
    import qiskit_ibm_runtime
    import pennylane_qiskit
    print('IBM backend dependencies already installed')
except ImportError:
    print('Installing qiskit-ibm-runtime and pennylane-qiskit...')
    try:
        _install('qiskit-ibm-runtime')
        _install('pennylane-qiskit')
        print('IBM backend dependencies installed successfully')
    except Exception as e:
        print(f'  IBM backend install failed ({e})')

# Confirm versions
import pennylane as qml
import torch
import numpy as np

print(f'PennyLane : {qml.__version__}')
print(f'PyTorch   : {torch.__version__}')
print(f'CUDA avail: {torch.cuda.is_available()}')
print(f'NumPy     : {np.__version__}')

try:
    import bloqade.analog as _bloqade
    print(f'Bloqade   : {_bloqade.__version__} (Aquila available)')
except ImportError:
    print('Bloqade   : NOT installed')

try:
    import qiskit_ibm_runtime
    print(f'Qiskit Runtime : {qiskit_ibm_runtime.__version__}')
except ImportError:
    print('Qiskit Runtime : NOT installed')

# Add project root
import sys
from pathlib import Path
PROJECT_ROOT = Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f'\nProject root: {PROJECT_ROOT}')

---
## Cell 2 - Configuration

In [ ]:
from config import *

# Phase 3 overrides
RUN_MODE = 'full'
N_QUBITS_PRIMARY = 9
J_STAR = 0.3
H_STAR = 0.8
P_STAR = 0.0
TOPOLOGY = 'chain'
USE_REUPLOADING = True
WARM_START_SOURCE = 'esn'

SWEEP_STEPS = 300 if RUN_MODE == 'smoke' else None

# Hardware backend selection for Cell 13
USE_SIMULATION = True
USE_AQUILA = False
USE_IBM_EAGLE = False

IBM_BACKEND_NAME = 'ibm_eagle'

print(f'Run mode         : {RUN_MODE}')
print(f'Qubits           : {N_QUBITS_PRIMARY}')
print(f'J*, h*           : {J_STAR}, {H_STAR}')
print(f'Data reuploading : {USE_REUPLOADING}')
print(f'Hardware backend : simulation={USE_SIMULATION}, aquila={USE_AQUILA}, ibm={USE_IBM_EAGLE}')

---
## Cell 3 - Data download

In [ ]:
from data.downloader import download_all

raw_paths = download_all()
print(f'{len(raw_paths)} year files ready.')

---
## Cell 4 - Parse, clean, and inspect

In [ ]:
from data.parser import load_and_merge

df = load_and_merge(raw_paths)

print(f'Timesteps  : {len(df)}')
print(f'Date range : {df.index.min()} -> {df.index.max()}')
print('\nNaN check (must all be 0%):')
for col in TARGETS:
    pct = df[col].isna().mean() * 100
    flag = 'OK' if pct == 0 else f'WARN {pct:.1f}%'
    print(f'  {col:<20} {flag}')

print('\nDescriptive stats:')
display(df[TARGETS].describe().round(2))

---
## Cell 5 - Preprocessing: shared PCA + windowing

In [ ]:
from preprocessing.pipeline import WeatherPreprocessor

if RUN_MODE == 'smoke':
    df_use = df.iloc[:500]
else:
    df_use = df

prep = WeatherPreprocessor(df_use)
datasets = prep.build_all()
prep.save(datasets)

for h, ds in datasets.items():
    print(ds.summary())

ds6 = datasets[6]
ds24 = datasets[24]
print('\nDatasets ready.')

---
## Cell 6 - Classical baselines

In [ ]:
from baselines.classical import (run_persistence, run_arima,
                                  run_esn, run_rnn, ARIMA_AVAILABLE, TORCH_AVAILABLE)
import time

baseline_results = {}
fitted_esn = None
fitted_rnn = {}
X_train_warm = None

if not ARIMA_AVAILABLE:
    print('WARNING: ARIMA backend not installed')
if not TORCH_AVAILABLE:
    print('WARNING: PyTorch not installed')

# Persistence
r = run_persistence(ds6.y_val, ds6.y_test, ds6.X_val, ds6.X_test, window=WINDOW_SIZE)
baseline_results['persistence'] = r
print(f'Persistence test RMSE: {r.test_rmse.mean():.4f}')

# ARIMA
r = run_arima(ds6.y_train, ds6.y_val, ds6.y_test, TARGETS)
if r:
    baseline_results['arima'] = r
    print(f'ARIMA test RMSE: {r.test_rmse.mean():.4f}')

# ESN
t0 = time.time()
r_esn, fitted_esn = run_esn(ds6.X_train, ds6.y_train, ds6.X_val, ds6.y_val, ds6.X_test, ds6.y_test)
baseline_results['esn'] = r_esn
print(f'ESN test RMSE: {r_esn.test_rmse.mean():.4f} [{time.time()-t0:.1f}s]')

# LSTM
t0 = time.time()
r, wrapper = run_rnn(ds6.X_train, ds6.y_train, ds6.X_val, ds6.y_val, ds6.X_test, ds6.y_test, window=WINDOW_SIZE, model_type='lstm')
if r:
    baseline_results['lstm'] = r
    fitted_rnn['lstm'] = wrapper
    print(f'LSTM test RMSE: {r.test_rmse.mean():.4f} [{time.time()-t0:.1f}s]')

# GRU
t0 = time.time()
r, wrapper = run_rnn(ds6.X_train, ds6.y_train, ds6.X_val, ds6.y_val, ds6.X_test, ds6.y_test, window=WINDOW_SIZE, model_type='gru')
if r:
    baseline_results['gru'] = r
    fitted_rnn['gru'] = wrapper
    print(f'GRU test RMSE: {r.test_rmse.mean():.4f} [{time.time()-t0:.1f}s]')

# Warm-start source - aligned with QRC warmup (QRC_WARMUP=20)
# Note: ESN uses ESN_WARMUP=50 which is > QRC_WARMUP=20
# The warm-start features are already trimmed by ESN's fit() method
# We need to ensure they match the QRC output shape
if WARM_START_SOURCE == 'esn' and fitted_esn is not None:
    X_train_warm = fitted_esn.get_reservoir_states(ds6.X_train)
    print(f'ESN warm states shape: {X_train_warm.shape}')
elif WARM_START_SOURCE in ('lstm', 'gru') and fitted_rnn.get(WARM_START_SOURCE) is not None:
    X_train_warm = fitted_rnn[WARM_START_SOURCE].get_hidden_states(ds6.X_train)
    print(f'RNN warm states shape: {X_train_warm.shape}')
print(f'Warm-start source: {WARM_START_SOURCE}')

---
## Cell 7 - Encoding ablation

In [ ]:
from reservoir.quantum_reservoir import encoding_ablation

encoding_results = encoding_ablation(
    X_train=ds6.X_train, y_train=ds6.y_train,
    X_val=ds6.X_val, y_val=ds6.y_val,
    n_qubits=N_QUBITS_PRIMARY, J=J_STAR, h=H_STAR,
    max_steps=SWEEP_STEPS, out_dir=RESULTS
)

print(f'Encoding ablation (n={N_QUBITS_PRIMARY}):')
for enc, rmse in encoding_results.items():
    print(f'  {enc}: val_rmse = {rmse:.4f}')

---
## Cell 8 - QRC training

In [ ]:
from reservoir.quantum_reservoir import IsingQRC
from readout.ridge_readout import RidgeReadout
import json, pickle, time

qrc_results = {}
out_h6 = RESULTS / 'h6'
out_h6.mkdir(parents=True, exist_ok=True)

for label, use_warm in [('cold_start_qrc', False), ('warm_start_qrc', True)]:
    print(f'\n--- Training: {label} ---')
    t0 = time.time()
    try:
        qrc = IsingQRC(
            n_qubits=N_QUBITS_PRIMARY, J=J_STAR, h=H_STAR,
            topology=TOPOLOGY, noise_rate=P_STAR,
            use_data_reuploading=USE_REUPLOADING,
            hardware_backend='simulation'
        )

        H_train = qrc.run_sequence(ds6.X_train, verbose=True)
        H_val = qrc.run_sequence(ds6.X_val)
        H_test = qrc.run_sequence(ds6.X_test)

        n_tr = min(len(H_train), len(ds6.y_train))
        n_vl = min(len(H_val), len(ds6.y_val))
        n_ts = min(len(H_test), len(ds6.y_test))

        readout = RidgeReadout(
            target_names=TARGETS,
            warm_start=(use_warm and X_train_warm is not None)
        )
        
        # Align warm-start features with QRC output
        # H_train has shape (T - QRC_WARMUP, 2*n_qubits) after discarding first 20 steps
        # X_train_warm may have a different length depending on the classical model's warmup
        X_warm_for_fit = None
        if use_warm and X_train_warm is not None:
            # Use the first n_tr rows of warm-start features (aligned with QRC output)
            X_warm_for_fit = X_train_warm[:n_tr]
            print(f'Warm-start alignment: QRC output={len(H_train)}, warm features={len(X_train_warm)}, using={n_tr}')
        
        best = readout.fit(
            H_train[:n_tr], ds6.y_train[:n_tr],
            H_val[:n_vl], ds6.y_val[:n_vl],
            X_train_warm_start=X_warm_for_fit
        )
        readout.save_selection_log(out_h6)

        pred_test = best.predict(H_test[:n_ts])
        pred_val = best.predict(H_val[:n_vl])

        class _R:
            def __init__(self, pv, pt):
                self.y_pred_val = pv
                self.y_pred_test = pt
        qrc_results[label] = _R(pred_val, pred_test)

        best.save(out_h6 / f'{label}_readout.pkl')
        cfg = qrc.get_config()
        cfg.update({'horizon_hours': 6, 'readout_strategy': best.strategy,
                    'warm_start': use_warm, 'warm_start_source': WARM_START_SOURCE,
                    'shared_pca': USE_SHARED_PCA,
                    'val_rmse_mean': float(best.val_rmse_mean),
                    'wall_clock_s': round(time.time()-t0, 1)})
        with open(out_h6 / f'{label}_config.json', 'w') as f:
            json.dump(cfg, f, indent=2)

        print(f'  Done. Strategy={best.strategy} val_rmse={best.val_rmse_mean:.4f} [{time.time()-t0:.0f}s]')

    except Exception as e:
        import traceback
        print(f'  FAILED: {e}')
        traceback.print_exc()

---
## Cell 9 - Qubit scaling study

In [ ]:
from experiments.sweeps import qubit_scaling_study

df_scaling = qubit_scaling_study(
    ds6.X_train, ds6.y_train,
    ds6.X_val, ds6.y_val,
    J=J_STAR, h=H_STAR, p=P_STAR,
    qubit_counts=QUBIT_COUNTS,
    use_data_reuploading=USE_REUPLOADING
)

print('\nQubit scaling results:')
display(df_scaling[['n_qubits', 'feature_dim', 'val_rmse']].to_string(index=False))

---
## Cell 10 - Full benchmark table

In [ ]:
from evaluation.metrics import build_results_table

all_results = {**baseline_results, **qrc_results}

print('=== 6-HOUR HORIZON ===')
df_6h = build_results_table(
    results=all_results,
    y_true_val=ds6.y_val,
    y_true_test=ds6.y_test,
    target_names=TARGETS,
    horizon_hours=6,
    out_dir=RESULTS
)
display(df_6h)

print('\n=== 24-HOUR HORIZON ===')
df_24h = build_results_table(
    results=baseline_results,
    y_true_val=ds24.y_val,
    y_true_test=ds24.y_test,
    target_names=TARGETS,
    horizon_hours=24,
    out_dir=RESULTS
)
display(df_24h)

---
## Cell 11 - Noise sweep

In [ ]:
from experiments.sweeps import noise_sweep

p_star, df_noise = noise_sweep(
    ds6.X_train, ds6.y_train,
    ds6.X_val, ds6.y_val,
    J=J_STAR, h=H_STAR, n_qubits=N_QUBITS_PRIMARY,
    use_data_reuploading=USE_REUPLOADING
)

print(f'\nOptimal noise rate p* = {p_star}')
display(df_noise)

---
## Cell 12 - Shot budget ablation

In [ ]:
from experiments.sweeps import shot_ablation

df_shots = shot_ablation(
    ds6.X_train, ds6.y_train,
    ds6.X_val, ds6.y_val,
    J=J_STAR, h=H_STAR, p=P_STAR, n_qubits=N_QUBITS_PRIMARY,
    use_data_reuploading=USE_REUPLOADING
)

print('Shot ablation results:')
display(df_shots)

---
## Cell 13 - Real-hardware validation

**Important:** This cell validates the selected config on real hardware.
Set `HARDWARE_BACKEND` below to choose the backend.

### Backend Options:

1. **Simulation** (Free) - `HARDWARE_BACKEND = 'simulation'`
2. **QuEra Aquila** (Real QPU) - `HARDWARE_BACKEND = 'aquila'`
3. **IBM Eagle** (Real QPU) - `HARDWARE_BACKEND = 'ibm'`

### IBM Eagle Setup (required for `HARDWARE_BACKEND = 'ibm'`):

1. Install dependencies (already done in Cell 1):
   ```
   pip install qiskit-ibm-runtime pennylane-qiskit
   ```

2. Save your IBM Quantum account token:
   ```python
   from qiskit_ibm_runtime import QiskitRuntimeService
   QiskitRuntimeService.save_account(
       channel="ibm_quantum",
       token="YOUR_TOKEN_HERE"
   )
   ```

3. Set `IBM_BACKEND_NAME` in Cell 2 (default: 'ibm_eagle')

**Note:** The IBM backend uses the existing circuit from `quantum_reservoir.py`
which runs unchanged against IBM Eagle via `pennylane-qiskit`.

In [ ]:
# Hardware backend selection - set EXACTLY ONE to True
HARDWARE_BACKEND = 'simulation'  # 'simulation' | 'aquila' | 'ibm'
HW_N_STEPS = 10                  # Keep small for hardware runs

import os

if HARDWARE_BACKEND == 'ibm':
    print('IBM Eagle backend selected')
    print(f'Backend name: {os.environ.get("QRC_IBM_BACKEND", IBM_BACKEND_NAME)}')
    print('\nVerifying IBM dependencies...')
    
    try:
        from qiskit_ibm_runtime import QiskitRuntimeService
        print('  qiskit-ibm-runtime: OK')
        
        # Check if token is available
        if os.environ.get('QRC_IBM_BACKEND') or IBM_BACKEND_NAME:
            print('  Backend configuration: OK')
        else:
            print('  WARNING: No IBM backend configured!')
            print('  Set IBM_BACKEND_NAME in Cell 2')
    except ImportError as e:
        print(f'  Missing dependency: {e}')
    
elif HARDWARE_BACKEND == 'aquila':
    print('Aquila backend selected')
    print(f'AQUILA_SUBMIT_TARGET: {os.environ.get("AQUILA_SUBMIT_TARGET", "local_emulator")}')
else:
    print('Simulation backend - free, no QPU credits used')

print(f'\nHW_N_STEPS: {HW_N_STEPS}')

from scripts.hardware_validation import run_hardware_validation

try:
    hw_result = run_hardware_validation(
        horizon=6, n_steps=HW_N_STEPS, backend=HARDWARE_BACKEND, mode='warm_start_qrc'
    )
    print('\n=== Hardware validation result ===')
    import json
    print(json.dumps(hw_result, indent=2))
    
    if HARDWARE_BACKEND != 'simulation' and HARDWARE_BACKEND in hw_result:
        delta = hw_result.get('sim_vs_hw_rmse_delta')
        print(f'\nSimulator vs {HARDWARE_BACKEND} RMSE delta: {delta:+.4f}')

except FileNotFoundError as e:
    print(f'Run Cell 8 first - need fitted config: {e}')
except ValueError as e:
    print(f'Configuration error: {e}')
except ImportError as e:
    print(f'Missing dependency: {e}')
except Exception as e:
    import traceback
    print(f'Hardware validation failed: {e}')
    traceback.print_exc()

---
## Cell 14 - Summary and output file list

In [ ]:
from config import RESULTS
import json

output_files = list(RESULTS.glob('*'))
print(f'Output files in {RESULTS}:')
for f in sorted(output_files):
    size_kb = f.stat().st_size // 1024 if f.exists() else 0
    print(f'  {f.name:<45} {size_kb:>6} KB')

print('\n=== KEY NUMBERS FOR WRITE-UP ===')
try:
    cfg = json.load(open(RESULTS / 'h6' / 'warm_start_qrc_config.json'))
    print(f'QRC config: n={cfg["n_qubits"]} J={cfg["J"]} h={cfg["h"]}')
    print(f'Encoding: {"data_reuploading" if cfg["use_data_reuploading"] else "standard"}')
    print(f'Circuit depth: {cfg["trotter_steps"]} Trotter steps')
    print(f'Readout: {cfg["readout_strategy"]}')
    print(f'Val RMSE: {cfg["val_rmse_mean"]:.4f}')
except FileNotFoundError:
    print('(Run Cell 8 first)')

if (RESULTS / 'hardware_validation.json').exists():
    hw = json.load(open(RESULTS / 'hardware_validation.json'))
    print(f'\nHardware validation: {hw.get("requested_backend")}')
else:
    print('\n(Run Cell 13 for hardware validation)')

print('\nPhase 3 checklist:')
checks = [
    ('results_h6.csv', (RESULTS / 'results_h6.csv').exists()),
    ('results_h24.csv', (RESULTS / 'results_h24.csv').exists()),
    ('qubit_scaling.csv', (RESULTS / 'qubit_scaling.csv').exists()),
    ('warm_start_qrc_config', (RESULTS / 'h6' / 'warm_start_qrc_config.json').exists()),
    ('hardware_validation.json', (RESULTS / 'hardware_validation.json').exists()),
]
for name, ok in checks:
    print(f'  {"[OK]" if ok else "[MISSING]"} {name}')